> **Legacy exploratory notebook (June 2025, first phase of the project).** Hyper-parameters were chosen on the test files and windows were split at random, so the scores it printed are not trustworthy estimates. It is kept for the record and is audited in `scripts/audit/` and `docs/legacy_audit.html`. Outputs were cleared (they contained participant file names); set `EEGBCI_LEGACY_DATA_DIR` for the helper `utils.py`.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
import torch.optim as optim
from glob import glob
import os
import mne
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense)
from tensorflow.keras.constraints import max_norm

In [ ]:
#read all file
all_files_path=glob('data/old/*.edf')
print(len(all_files_path))
print(all_files_path)

In [ ]:
scenario = [i for i in all_files_path if ("3_S07" in i)]
print("\n".join(scenario))
print(len(scenario))

In [ ]:
def read_data(file_path, channels=None, scenario=1):
    datax=mne.io.read_raw_edf(file_path,preload=True, verbose=False)

    if channels:
        # Ensure the channels exist before dropping to avoid errors
        existing_channels = datax.ch_names
        other_channels = [ch for ch in existing_channels if ch not in channels]
        if other_channels:
            print(f"Dropping channels: {other_channels}")
            datax.drop_channels(other_channels)
        else:
            print("No specified channels to drop were found in the data.")
   

    # datax=mne.io.read_raw_fif(file_path,preload=True)
    datax.set_eeg_reference(verbose=False)
    # epochs=mne.make_fixed_length_epochs(datax,duration=1,overlap=0)
    # datax.filter(8., 30., fir_design='firwin', n_jobs=8)
    # epochs=mne.make_fixed_length_epochs(datax,duration=0.2,overlap=0.1)

    data, times = datax[:, :]   
    times = times - times[0]
    if scenario == 1:
        """"
            neutre 0 - 240 X
            push 240 - 270 X
            neutre 270 - 290 X
            push 290 - 320 18
            neutre 320 - 340 X
            walk 340 - 370 X
            neutre 370 - 390 8
            walk 390 - 420 18
            neutre 420 - 480 /


            mask_train = [0, 308], []
        """
        total = int(len(times))
        NeutreTrain = 288*500
        PushTrain = 48*500
        WalkTrain = 48*500
        i = 0
        indxtrainNeutre = []
        indxtrainPush = []
        indxtrainWalk = []
        indxtestNeutre = []
        indxtestPush = []
        indxtestWalk = []
        for i in range(0, total): 
            if i < 240*500 or (i >= 270*500 and i < 290*500) or (i >= 320*500 and i < 340*500) or (i >= 370*500 and i < 390*500) or (i >= 420*500 and i < 480*500): #neutre
                if NeutreTrain > 0:
                    indxtrainNeutre.append(i)
                else:
                    indxtestNeutre.append(i)
                NeutreTrain -= 1
            elif (i >= 240*500 and i < 270*500) or (i >= 290*500 and i < 320*500) : #push
                if PushTrain > 0:
                    indxtrainPush.append(i)
                else:
                    indxtestPush.append(i)
                PushTrain -= 1
            elif (i >= 340*500 and i < 370*500) or (i >= 390*500 and i < 420*500)  : #walk
                if WalkTrain > 0:
                    indxtrainWalk.append(i)
                else:
                    indxtestWalk.append(i)
                WalkTrain -= 1
    elif scenario == 2:
        """"
            push 80
            walk 80
            neutre 20
            push 80 
            walk 80 
            neutre 20
            push 70 
            walk 70

        """
        total = int(len(times))
        NeutreTrain = 40*500 *0.8
        PushTrain =   230*500 *0.8
        WalkTrain =   230*500 *0.8
        i = 0
        indxtrainNeutre = []
        indxtrainPush = []
        indxtrainWalk = []
        indxtestNeutre = []
        indxtestPush = []
        indxtestWalk = []
        for i in range(0, total): 
            if i < 80*500 or (i >= 180*500 and i < 260*500) or (i >= 360*500 and i < 430*500) : #push
                if PushTrain > 0:
                    indxtrainPush.append(i)
                else:
                    indxtestPush.append(i)
                PushTrain -= 1
            elif (i >= 80*500 and i < 160*500) or (i >= 260*500 and i < 340*500) or (i >= 430*500 and i < 500*500)  : #walk
                if WalkTrain > 0:
                    indxtrainWalk.append(i)
                else:
                    indxtestWalk.append(i)
                WalkTrain -= 1
            else: #neutre
                if NeutreTrain > 0:
                    indxtrainNeutre.append(i)
                else:
                    indxtestNeutre.append(i)
                NeutreTrain -= 1

    # print("indxtrainNeutre",indxtrainNeutre)
    # print("indxtrainPush",indxtrainPush)
    # print("indxtrainWalk",indxtrainWalk)
    # print("indxtestNeutre",indxtestNeutre)
    # print("indxtestPush",indxtestPush)
    # print("indxtestWalk",indxtestWalk)


    

    epochs=datax.get_data(verbose=False)
    # keep channels names
    ch_names = datax.ch_names
    print(ch_names)
    return ch_names
    return epochs, (indxtrainNeutre,indxtrainPush,indxtrainWalk,indxtestNeutre,indxtestPush,indxtestWalk) 

In [ ]:

import glob
folder1 = glob.glob("<local-path>")
data1 = [read_data(f) for f in folder1]

folder2 = glob.glob("<local-path>")
data2 = [read_data(f) for f in folder2]

# length of each dataset
print("Length of each dataset:")
print("Data1: ", len(data1))
print("Data2: ", len(data2))
print("")

data_train = []
data_test = []
for person in data1 + data2:
    data, indeces = person
    data_train += [[data[indeces[0]], 0] for i in range(len(indeces))]
    data_test += [[data[indeces[3]], 0] for i in range(len(indeces))]
    

data_train = np.array(data_train, dtype=object)
data_test = np.array(data_test, dtype=object)


# count the number of each class in the train data
print("Train data:")
print("Neutre: ", len(data_train[data_train[:, 1] == 0]))
print("Push: ", len(data_train[data_train[:, 1] == 1]))
print("Walk: ", len(data_train[data_train[:, 1] == 2]))





- S05     80% 90s  20%
- S07    80% 432s  20%

totale 80% train 20% test
